# 05 — Backtest: RandomTrader (engine correctness check)

Expected: negative net P&L of roughly -total_fees over many random trades.

In [ ]:
from datetime import datetime, timezone
from src.layer1_research.backtesting.config import BacktestConfig
from src.layer1_research.backtesting.runner import BacktestRunner
from src.layer1_research.backtesting.strategies.examples.random_trader import (
    RandomTraderStrategy,
)
from src.layer1_research.backtesting.reporting.cli_report import print_report

In [ ]:
config = BacktestConfig(
    catalog_path="data/catalog",
    start=datetime(2024, 6, 1, tzinfo=timezone.utc),
    end=datetime(2024, 9, 1, tzinfo=timezone.utc),
    strategy_name="random_trader",
    starting_capital=10_000.0,
    data_mode="trade",
    fee_rate_bps=100,
    strategy_params={"seed": 42, "p_trade": 0.02, "trade_size": 5.0},
)

In [ ]:
result = BacktestRunner(config).run(RandomTraderStrategy)
print_report(result.metrics())

In [ ]:
result.plot_equity_curve()

In [ ]:
result.plot_pnl_histogram()

### Sanity check: final P&L should be roughly -total_fees

In [ ]:
net_pnl = float(result.equity_curve.iloc[-1]) - config.starting_capital
fees = result.metrics().total_fees
print(f"Net P&L:    ${net_pnl:,.2f}")
print(f"Total fees: ${fees:,.2f}")
if fees > 0:
    print(f"Ratio:      {net_pnl / -fees:.2f}   (ideal: ~1.0)")